### 文本測試

斷詞 × 表示法
- 斷詞方法（2）
    - jieba
    - CKIP
- 文字表示（3）
    - tf-idf
    - word2vec (mean)
    - fastText (mean)
- 分類器（1）
    - Logistic Regression（固定）

In [43]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from tqdm import tqdm
tqdm.disable = True

import time
import io
import contextlib

@contextlib.contextmanager
def silent():
    f_out, f_err = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(f_out), contextlib.redirect_stderr(f_err):
        yield

def timed(fn, *args, **kwargs):
    t0 = time.perf_counter()
    with silent():
        out = fn(*args, **kwargs)
    t1 = time.perf_counter()
    return out, (t1 - t0)

import os

def default_workers(reserve=1, max_workers=None):
    n = os.cpu_count() or 4
    w = max(1, n - reserve)
    if max_workers is not None:
        w = min(w, max_workers)
    return w

WORKERS = default_workers(reserve=2, max_workers=12)  # 例如最多用到 12


#### ✅開始抓資料+切分資料
- 資料切分：依 y 分層抽樣，各類各取 500 筆（共 1000 筆）。
- 標籤定義：以 score_overall ≥ 2 定義為 1，其餘為 0。

In [44]:
import sqlite3
import pandas as pd

def load_pilot_data(
    db_file: str,
    threshold: int = 3,
    n_per_class: int = 500,
    random_state: int = 42
) -> pd.DataFrame:
    """
    從 SQLite 讀取：
      - article_comments.comment_text (X)
      - eb_evaluation 最新一筆 score_overall (用來做 y)
    並做 pilot 分層抽樣：y=0/1 各取 n_per_class。

    y 定義：score_overall >= threshold -> 1 else 0
    """
    conn = sqlite3.connect(db_file)

    df = pd.read_sql_query("""
        SELECT 
            c.id AS comment_id,
            c.article_id,
            c.comment_index,
            c.comment_text,

            e.level,
            e.created_at AS eval_created_at,
            e.score_overall,
            e.main_strategy,
            e.main_strategy_detail,
            e.confidence,
            e.model_name,
            e.model_version,
            e.knowledge_base
        FROM article_comments c
        JOIN (
            SELECT comment_id, MAX(created_at) AS max_created_at
            FROM eb_evaluation
            WHERE comment_id IS NOT NULL
            GROUP BY comment_id
        ) latest
          ON latest.comment_id = c.id
        JOIN eb_evaluation e
          ON e.comment_id = latest.comment_id
         AND e.created_at = latest.max_created_at
    """, conn)

    conn.close()

    # 基本清理
    df = df.dropna(subset=["comment_text", "score_overall"]).copy()
    df["comment_text"] = df["comment_text"].astype(str)

    # 二元標籤
    df["y"] = (df["score_overall"] >= threshold).astype(int)

    # 分層抽樣：每類取 n_per_class（先打散再 head）
    df_pilot = (
        df.sample(frac=1, random_state=random_state)
          .groupby("y", sort=False)
          .head(n_per_class)
          .reset_index(drop=True)
    )

    return df_pilot

DB_FILE = r"D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"

df_pilot = load_pilot_data(DB_FILE, threshold=2, n_per_class=500)
print("df_pilot shape:", df_pilot.shape)
print(df_pilot["y"].value_counts())
print(df_pilot[["comment_id", "score_overall", "y", "comment_text"]].head(2))


df_pilot shape: (1000, 14)
y
0    500
1    500
Name: count, dtype: int64
                 comment_id  score_overall  y  \
0  6923bf32af113796ec892438            0.0  0   
1  6923bf32af113796ec8921f2            1.0  0   

                                        comment_text  
0  是要用網際網路開FB...\n\n然後改成電腦版的...應該就可以玩了..\n\n開心農場都...  
1                                    這和人情無關\n\n純粹想宰羊  


#### ✅斷詞


In [45]:
import re
import jieba
import os
import io
import contextlib

# ============ jieba ==============

def normalize_text(text: str) -> str:
    """
    最小清理：保留內容，只把多餘空白/換行收斂。
    （Pilot 先不要做太多清洗，避免把訊號洗掉）
    """
    if text is None:
        return ""
    text = str(text)
    # 把所有空白（含換行、tab）收斂成單一空白
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_jieba(text: str):
    if text is None:
        return []
    text = re.sub(r"\s+", " ", str(text)).strip()
    return [t for t in jieba.cut(text, cut_all=False) if t.strip()]

# ===== CKIP tokenizer（lazy load）=====
_CKIP_WS = None

def get_ckip_ws():
    global _CKIP_WS
    if _CKIP_WS is None:
        from ckip_transformers.nlp import CkipWordSegmenter

        f_out, f_err = io.StringIO(), io.StringIO()
        with contextlib.redirect_stdout(f_out), contextlib.redirect_stderr(f_err):
            _CKIP_WS = CkipWordSegmenter(model="bert-base")
    return _CKIP_WS

def tokenize_ckip(text: str):
    if text is None:
        return []
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return []

    ws = get_ckip_ws()

    # 同時吃掉 stdout + stderr（把 progress / load report / warning 全關）
    f_out, f_err = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(f_out), contextlib.redirect_stderr(f_err):
        tokens = ws([text])[0]

    return [t for t in tokens if t.strip()]


DB_FILE = r"D:/NTPU_class/paper/code/mobile01_clawler/ntpu_paper.sqlite"
df_pilot = load_pilot_data(DB_FILE, threshold=2, n_per_class=500)

import time

for i in range(3, 8):
    text = df_pilot.loc[i, "comment_text"]
    print(f"\n--- sample {i-2} ---")

    t0 = time.perf_counter()
    jieba_tokens = tokenize_jieba(text)
    t1 = time.perf_counter()

    t2 = time.perf_counter()
    ckip_tokens = tokenize_ckip(text)
    t3 = time.perf_counter()

    print(f"JIEBA ({t1 - t0:.4f}s):", jieba_tokens[:30])
    print(f"CKIP  ({t3 - t2:.4f}s):", ckip_tokens[:30])



--- sample 1 ---


BertForTokenClassification LOAD REPORT from: ckiplab/bert-base-chinese-ws
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


JIEBA (0.0006s): ['健身房', '不是', '給有', '錢', '人用', '的', '嗎', '我', '想', '健身房', '有', '專門', '的', '教練', '可以', '給意', '見', '可以', '提供', '適合個', '人', '的', '訓練表', '所以', '才', '會', '有', '健身房', '吧', '而且']
CKIP  (1.6390s): ['健身房', '不', '是', '給', '有錢人', '用', '的', '嗎', '我', '想', '健身房', '有', '專門', '的', '教練', '可以', '給', '意見', '可以', '提供', '適合', '個人', '的', '訓練表', '所以', '才', '會', '有', '健身房', '吧']

--- sample 2 ---
JIEBA (0.0008s): ['Ariouk', 'wrote', ':', '我', '原本', '都', '聽', '串流', 'Spo', '...', '(', '恕', '刪', ')', '串流', '除', '方便', '外', '，', '音質', '就', '…', '不', '強求', '了', 'cd', '可以', '自己', 'rip', '倒']
CKIP  (0.0531s): ['Ariouk', ' wrote: ', '我', '原本', '都', '聽', '串流', 'Spo...', '(', '恕', '刪', ')', '串流', '除', '方便', '外', '，', '音質', '就', '…', '不', '強求', '了', ' cd', '可以', '自己', 'rip', '倒是', '最近', '比較']

--- sample 3 ---
JIEBA (0.0002s): ['我', '之前', '都', '是', 'google', '釘', '選用', '一用', '而已', '沒', '想到', '有', '這些', '可以', '用']
CKIP  (0.0240s): ['我', '之前', '都', '是', 'google釘', '選用', '一', '用', '而已', '沒想到', '有', '這些', '

#### ✅ 文字表示

In [46]:
# pip install scikit-learn gensim jieba

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def make_tokenized_corpus(df, text_col, tokenizer):
    # 斷詞後用空白串接，給 TF-IDF 用
    return [" ".join(tokenizer(t)) for t in df[text_col].astype(str).tolist()]

def make_tokens(df, text_col, tokenizer):
    # 給 w2v / fastText 用（list of tokens）
    return [tokenizer(t) for t in df[text_col].astype(str).tolist()]

# ---------- 1) TF-IDF ----------
def encode_tfidf(df, tokenizer, text_col="comment_text", max_features=3000, min_df=2):
    corpus = make_tokenized_corpus(df, text_col=text_col, tokenizer=tokenizer)
    vec = TfidfVectorizer(max_features=max_features, min_df=min_df)
    X = vec.fit_transform(corpus)
    return X, vec

def encode_tfidf_from_tokens(tokens, max_features=3000, min_df=2, ngram_range=(1,1)):
    corpus = [" ".join(t) for t in tokens]
    vec = TfidfVectorizer(
        max_features=max_features,
        min_df=min_df,
        ngram_range=ngram_range,
        dtype=np.float32
    )
    X = vec.fit_transform(corpus)
    return X, vec

# mean pooling 共用
def mean_pool(tokens_list, wv, vector_size: int):
    X = np.zeros((len(tokens_list), vector_size), dtype=np.float32)
    coverage = []
    for i, toks in enumerate(tokens_list):
        vecs = [wv[w] for w in toks if w in wv]
        if vecs:
            X[i] = np.mean(vecs, axis=0)
            coverage.append(len(vecs) / max(len(toks), 1))
        else:
            coverage.append(0.0)
    return X, float(np.mean(coverage))

# ---------- 2) Word2Vec(mean) ----------
def encode_w2v_mean(
    df, tokenizer, text_col="comment_text",
    vector_size=100, window=5, min_count=2, workers=None, epochs=10
):
    from gensim.models import Word2Vec

    if workers is None:
        workers = WORKERS

    tokens = make_tokens(df, text_col=text_col, tokenizer=tokenizer)
    model = Word2Vec(
        sentences=tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers
    )
    model.train(tokens, total_examples=len(tokens), epochs=epochs)

    X, cov = mean_pool(tokens, model.wv, vector_size)
    return X, model, cov

def encode_w2v_mean_from_tokens(
    tokens,
    vector_size=100, window=5, min_count=2, workers=None, epochs=10
):
    from gensim.models import Word2Vec
    if workers is None:
        workers = WORKERS

    model = Word2Vec(
        sentences=tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers
    )
    model.train(tokens, total_examples=len(tokens), epochs=epochs)

    X, cov = mean_pool(tokens, model.wv, vector_size)
    return X, model, cov


# ---------- 3) FastText(mean) ----------
def encode_fasttext_mean(
    df, tokenizer, text_col="comment_text",
    vector_size=100, window=5, min_count=2, workers=None, epochs=10
):
    from gensim.models import FastText

    if workers is None:
        workers = WORKERS

    tokens = make_tokens(df, text_col=text_col, tokenizer=tokenizer)
    model = FastText(
        sentences=tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers
    )
    model.train(tokens, total_examples=len(tokens), epochs=epochs)

    X, cov = mean_pool(tokens, model.wv, vector_size)
    return X, model, cov

def encode_fasttext_mean_from_tokens(
    tokens,
    vector_size=100, window=5, min_count=2, workers=None, epochs=10
):
    from gensim.models import FastText
    if workers is None:
        workers = WORKERS

    model = FastText(
        sentences=tokens,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers
    )
    model.train(tokens, total_examples=len(tokens), epochs=epochs)

    X, cov = mean_pool(tokens, model.wv, vector_size)
    return X, model, cov

# ---------- 展示用（最小） ----------
def show_encoding_summary(name, X, seconds=None, extra=None, tfidf_vec=None, topn=10):
    msg = f"[{name}] shape={X.shape}"
    if seconds is not None:
        msg += f" time={seconds:.2f}s"
    print(msg)

    if tfidf_vec is not None:
        feats = tfidf_vec.get_feature_names_out()[:topn]
        print("  tfidf_features_head:", list(feats))

    if extra:
        for k, v in extra.items():
            if isinstance(v, float):
                print(f"  {k}={v:.3f}")
            else:
                print(f"  {k}={v}")


In [47]:
# 先斷詞一次（cache）
tokens_j = make_tokens(df_pilot, "comment_text", tokenize_jieba)
tokens_c = make_tokens(df_pilot, "comment_text", tokenize_ckip)

# ===== jieba =====
(X_tfidf_j, vec_j), t_tfidf_j = timed(encode_tfidf_from_tokens, tokens_j)
show_encoding_summary("jieba + tfidf", X_tfidf_j, seconds=t_tfidf_j, tfidf_vec=vec_j)

(X_w2v_j, w2v_j, cov_w2v_j), t_w2v_j = timed(
    encode_w2v_mean_from_tokens, tokens_j, epochs=5, workers=WORKERS
)
show_encoding_summary("jieba + w2v(mean)", X_w2v_j, seconds=t_w2v_j, extra={"coverage": cov_w2v_j})

(X_ft_j, ft_j, cov_ft_j), t_ft_j = timed(
    encode_fasttext_mean_from_tokens, tokens_j, epochs=5, workers=WORKERS
)
show_encoding_summary("jieba + fastText(mean)", X_ft_j, seconds=t_ft_j, extra={"coverage": cov_ft_j})

# ===== ckip =====
(X_tfidf_c, vec_c), t_tfidf_c = timed(encode_tfidf_from_tokens, tokens_c)
show_encoding_summary("ckip + tfidf", X_tfidf_c, seconds=t_tfidf_c, tfidf_vec=vec_c)

(X_w2v_c, w2v_c, cov_w2v_c), t_w2v_c = timed(
    encode_w2v_mean_from_tokens, tokens_c, epochs=5, workers=WORKERS
)
show_encoding_summary("ckip + w2v(mean)", X_w2v_c, seconds=t_w2v_c, extra={"coverage": cov_w2v_c})

(X_ft_c, ft_c, cov_ft_c), t_ft_c = timed(
    encode_fasttext_mean_from_tokens, tokens_c, epochs=5, workers=WORKERS
)
show_encoding_summary("ckip + fastText(mean)", X_ft_c, seconds=t_ft_c, extra={"coverage": cov_ft_c})


[jieba + tfidf] shape=(1000, 3000) time=0.05s
  tfidf_features_head: ['000', '01', '04', '07', '08', '10', '100', '1000', '11', '12']
[jieba + w2v(mean)] shape=(1000, 100) time=0.49s
  coverage=0.847
[jieba + fastText(mean)] shape=(1000, 100) time=1.69s
  coverage=1.000
[ckip + tfidf] shape=(1000, 2892) time=0.03s
  tfidf_features_head: ['000', '01', '04', '07', '08', '09', '10', '100', '1000', '100萬']
[ckip + w2v(mean)] shape=(1000, 100) time=0.47s
  coverage=0.883
[ckip + fastText(mean)] shape=(1000, 100) time=1.31s
  coverage=1.000


- TF-IDF 計算速度最快，但在短文本中容易被「數字與非語意詞」主導。
- jieba 斷詞在效率上明顯優於 CKIP，後者雖較穩定但時間成本極高。
- Word2Vec(mean) 在短文本中仍存在詞彙未被表示（OOV）的問題。
- fastText(mean) 透過 subword 機制，能完整涵蓋所有詞彙（coverage = 1.0），較適合論壇短留言與新詞情境。


1. [ TF-IDF ]
   - shape=(1000, 3000) 代表
   - 1000：樣本數（500 正例 + 500 負例）
   - 3000：最多取 3000 個詞彙作為特徵（max_features=3000）
   - tfidf_features_head：只取前 10 個來看 (不是數量最多、不是最重要、不是權重最高)

2. [ Word2Vec(mean) ]
   - 先學每個詞的向量（word embedding）
   - 再把一則留言中所有詞向量「取平均」，得到一句話的向量
   - shape=(1000, 100) 代表每則留言被表示成 100 維的語意向量

3. [ fastText(mean) ]
   - 與 Word2Vec 類似，但引入 subword（字元 n-gram）
   - 即使整個詞沒看過，也能用「詞的一部分」組合出向量

coverage = 有向量的詞數 / 總詞數


#### 查看哪個詞出現機率比較高!

In [48]:
import numpy as np

def get_top_tfidf_terms(X, vec, topn=20):
    # X: tf-idf matrix
    # vec: fitted TfidfVectorizer
    avg_tfidf = X.mean(axis=0).A1
    top_idx = avg_tfidf.argsort()[::-1][:topn]
    feature_names = vec.get_feature_names_out()
    return [(feature_names[i], avg_tfidf[i]) for i in top_idx]
top_tfidf_jieba = get_top_tfidf_terms(X_tfidf_j, vec_j, topn=20)
top_tfidf_jieba

[('wrote', np.float32(0.042327862)),
 ('自己', np.float32(0.023556538)),
 ('可以', np.float32(0.020806825)),
 ('真的', np.float32(0.019211952)),
 ('就是', np.float32(0.017666489)),
 ('不是', np.float32(0.01607495)),
 ('如果', np.float32(0.014542133)),
 ('知道', np.float32(0.013934286)),
 ('什麼', np.float32(0.013029874)),
 ('台灣', np.float32(0.012675131)),
 ('一個', np.float32(0.011679349)),
 ('應該', np.float32(0.011014274)),
 ('所以', np.float32(0.010971107)),
 ('不要', np.float32(0.010913443)),
 ('因為', np.float32(0.010884163)),
 ('還是', np.float32(0.010774741)),
 ('覺得', np.float32(0.0107651185)),
 ('問題', np.float32(0.010686929)),
 ('可能', np.float32(0.010454278)),
 ('一下', np.float32(0.01015462))]

#### ✅分類器

In [49]:
# 先準備：y、切 train/test（固定）
from sklearn.model_selection import train_test_split

y = df_pilot["y"].values

train_idx, test_idx = train_test_split(
    df_pilot.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

df_train = df_pilot.loc[train_idx].reset_index(drop=True)
df_test  = df_pilot.loc[test_idx].reset_index(drop=True)

y_train = df_train["y"].values
y_test  = df_test["y"].values

In [50]:
# 估工具：跑 LR + 回傳指標 
import numpy as np
from gensim.models import Word2Vec
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from gensim.models import FastText

def eval_lr(X_train, X_test, y_train, y_test, name=""):
    t0 = time.perf_counter()

    clf = LogisticRegression(max_iter=2000, solver="liblinear")
    clf.fit(X_train, y_train)

    pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test)[:, 1]

    t1 = time.perf_counter()

    return {
        "model": name,
        "acc": accuracy_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "auc": roc_auc_score(y_test, proba),
        "time_sec": t1 - t0
    }

# TF-IDF：train fit，test transform（避免 leakage）
def tfidf_train_test(df_train, df_test, tokenizer=None, max_features=3000, min_df=2,
                     tokens_train=None, tokens_test=None):
    # ✅ 若已提供 tokens，就不要再 tokenize
    if tokens_train is None:
        tokens_train = make_tokens(df_train, "comment_text", tokenizer)
    if tokens_test is None:
        tokens_test = make_tokens(df_test, "comment_text", tokenizer)

    # tokens -> string corpus
    corpus_train = [" ".join(t) for t in tokens_train]
    corpus_test  = [" ".join(t) for t in tokens_test]

    vec = TfidfVectorizer(max_features=max_features, min_df=min_df)
    X_train = vec.fit_transform(corpus_train)
    X_test  = vec.transform(corpus_test)
    return X_train, X_test, vec

# w2v/fastText：只用 train 訓練，再對 test 做 mean pooling
def encode_mean_from_tokens(tokens, model_wv, vector_size=100):
    X, cov = mean_pool(tokens, model_wv, vector_size)
    return X, cov

def w2v_train_test(df_train, df_test, tokenizer=None, vector_size=100, window=5,
                   min_count=2, workers=4, epochs=10,
                   tokens_train=None, tokens_test=None):

    if tokens_train is None:
        tokens_train = make_tokens(df_train, "comment_text", tokenizer)
    if tokens_test is None:
        tokens_test = make_tokens(df_test, "comment_text", tokenizer)

    model = Word2Vec(
        sentences=tokens_train,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers  # ✅ 多核心在這裡
    )
    model.train(tokens_train, total_examples=len(tokens_train), epochs=epochs)

    X_train, cov_train = encode_mean_from_tokens(tokens_train, model.wv, vector_size)
    X_test,  cov_test  = encode_mean_from_tokens(tokens_test,  model.wv, vector_size)

    return X_train, X_test, model, cov_train, cov_test


# fastText train/test
def ft_train_test(df_train, df_test, tokenizer=None, vector_size=100, window=5,
                  min_count=2, workers=4, epochs=10,
                  tokens_train=None, tokens_test=None):

    if tokens_train is None:
        tokens_train = make_tokens(df_train, "comment_text", tokenizer)
    if tokens_test is None:
        tokens_test = make_tokens(df_test, "comment_text", tokenizer)

    model = FastText(
        sentences=tokens_train,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers  # ✅ 多核心在這裡
    )
    model.train(tokens_train, total_examples=len(tokens_train), epochs=epochs)

    X_train, cov_train = encode_mean_from_tokens(tokens_train, model.wv, vector_size)
    X_test,  cov_test  = encode_mean_from_tokens(tokens_test,  model.wv, vector_size)

    return X_train, X_test, model, cov_train, cov_test

In [51]:
import pandas as pd
import time
import os

# ===== 你可以調這裡 =====
N_WORKERS = max(1, (os.cpu_count() or 4) - 1)
W2V_EPOCHS = 10
FT_EPOCHS  = 10

results = []

def run_and_record(name, prep_sec, build_fn, extra_kv=None):
    t0 = time.perf_counter()
    out = build_fn()
    feat_sec = time.perf_counter() - t0

    # out 可以是 (Xtr, Xte) 或 (Xtr, Xte, extra)
    if isinstance(out, tuple) and len(out) == 3:
        Xtr, Xte, extra = out
    else:
        Xtr, Xte = out
        extra = None

    r = eval_lr(Xtr, Xte, y_train, y_test, name)
    r["prep_sec"] = prep_sec
    r["feat_sec"] = feat_sec
    r["total_sec"] = r["prep_sec"] + r["feat_sec"] + r["time_sec"]

    if isinstance(extra_kv, dict):
        r.update(extra_kv)
    if isinstance(extra, dict):
        r.update(extra)

    results.append(r)


# =========================
# 1) 先做一次斷詞（cache tokens）
# =========================
t0 = time.perf_counter()
jieba_tr = make_tokens(df_train, "comment_text", tokenize_jieba)
jieba_te = make_tokens(df_test,  "comment_text", tokenize_jieba)
jieba_prep = time.perf_counter() - t0

t0 = time.perf_counter()
ckip_tr = make_tokens(df_train, "comment_text", tokenize_ckip)
ckip_te = make_tokens(df_test,  "comment_text", tokenize_ckip)
ckip_prep = time.perf_counter() - t0


# =========================
# 2) TF-IDF（✅補回來）
# =========================
run_and_record(
    "jieba + tfidf",
    jieba_prep,
    lambda: tfidf_train_test(df_train, df_test, tokens_train=jieba_tr, tokens_test=jieba_te)[:2]
)

run_and_record(
    "ckip + tfidf",
    ckip_prep,
    lambda: tfidf_train_test(df_train, df_test, tokens_train=ckip_tr, tokens_test=ckip_te)[:2]
)


# =========================
# 3) Word2Vec(mean)
# =========================
def build_w2v(tokens_tr, tokens_te):
    Xtr, Xte, model, cov_tr, cov_te = w2v_train_test(
        df_train, df_test,
        tokens_train=tokens_tr,
        tokens_test=tokens_te,
        workers=N_WORKERS,
        epochs=W2V_EPOCHS
    )
    return Xtr, Xte, {"cov_train": cov_tr, "cov_test": cov_te}

run_and_record("jieba + w2v(mean)", jieba_prep, lambda: build_w2v(jieba_tr, jieba_te))
run_and_record("ckip + w2v(mean)",  ckip_prep,  lambda: build_w2v(ckip_tr,  ckip_te))


# =========================
# 4) fastText(mean)
# =========================
def build_ft(tokens_tr, tokens_te):
    Xtr, Xte, model, cov_tr, cov_te = ft_train_test(
        df_train, df_test,
        tokens_train=tokens_tr,
        tokens_test=tokens_te,
        workers=N_WORKERS,
        epochs=FT_EPOCHS
    )
    return Xtr, Xte, {"cov_train": cov_tr, "cov_test": cov_te}

run_and_record("jieba + fastText(mean)", jieba_prep, lambda: build_ft(jieba_tr, jieba_te))
run_and_record("ckip + fastText(mean)",  ckip_prep,  lambda: build_ft(ckip_tr,  ckip_te))


# =========================
# 5) 結果表
# =========================
df_res = pd.DataFrame(results)

for c in ["acc","f1","auc","prep_sec","feat_sec","time_sec","total_sec","cov_train","cov_test"]:
    if c in df_res.columns:
        df_res[c] = df_res[c].round(3)

df_res


,model,acc,f1,auc,time_sec,prep_sec,feat_sec,total_sec,cov_train,cov_test
0,jieba + tfidf,0.635,0.644,0.713,0.002,0.538,0.044,0.584,NaN,NaN
1,ckip + tfidf,0.640,0.647,0.682,0.002,75.834,0.040,75.877,NaN,NaN
2,jieba + w2v(mean),0.555,0.578,0.569,0.007,0.538,0.576,1.120,0.835,0.768
3,ckip + w2v(mean),0.540,0.593,0.546,0.005,75.834,0.536,76.376,0.873,0.806
4,jieba + fastText(mean),0.545,0.569,0.569,0.006,0.538,1.359,1.902,1.000,1.000
5,ckip + fastText(mean),0.540,0.578,0.557,0.004,75.834,1.480,77.318,1.000,1.000


其實結果真的有點奇怪嗎
但也是還好嗎

GPT叫我繼續做<br>
1️⃣ char TF-IDF (2,4) + LR（成功率最高）<br>
2️⃣ word TF-IDF (1,2) + LR<br>
3️⃣ word TF-IDF + LinearSVC<br><br>
就是 一次看： 2 個字、3 個字、4 個字